In [ ]:
import numpy as np
from scipy.optimize import curve_fit

def fit_q(W, R):
    W, R = np.asarray(W, float), np.asarray(R, float)

    # Method 1: log-space regression through the origin (ln R = q ln W)
    x, y = np.log(W), np.log(R)
    q_log = np.sum(x * y) / np.sum(x ** 2)

    # Method 2: nonlinear least squares on the original scale (R = W**q)
    q_nls, cov = curve_fit(lambda w, q: w ** q, W, R, p0=[1.0])
    se = np.sqrt(np.diag(cov))[0]
    return q_log, q_nls[0], se


# W = relative water availability, R = relative growth (stressed / control)
"Thymus vulgaris"
W = [0.75, 0.50]
R = [0.938, 0.706]
q_log, q_nls, se = fit_q(W, R)
print(f"q (log) = {q_log:.2f}, q (NLS) = {q_nls:.2f} ± {se:.2f}")

q (log) = 0.46, q (NLS) = 0.44 ± 0.11


In [ ]:
"Festuca arundinacea"
W = [0.8, 0.65, 0.5, 0.35]
B = [0.14, 0.113, 0.073, 0.067]
R = []
for i in B:
    R.append(float(i)/0.163)

q_log, q_nls, se = fit_q(W, R)
print(f"q (log) = {q_log:.2f}, q (NLS) = {q_nls:.2f} ± {se:.2f}")


q (log) = 0.93, q (NLS) = 0.93 ± 0.09


In [ ]:
"""Estimate the drought-sensitivity exponent q in R = W**q from WUE and water supplied (Amburana cearensis).

Assumes WUE = biomass / water, so biomass ~ WUE * water.
Dataset: Amburana cearensis (WUE at 50/75/100 % of ETo).
"""
import numpy as np
from scipy.optimize import curve_fit

# --- inputs ---------------------------------------------------------------
water = np.array([50.0, 75.0, 100.0])      # % of ETo (W1, W2, W3)
wue = np.array([0.249, 0.209, 0.240])      # g biomass per L water (mean)
wue_sd = np.array([0.016, 0.016, 0.013])   # SD, used only for the uncertainty check
ref = -1                                   # index of control (100 %)

# --- relative water and relative growth -----------------------------------
W = water / water[ref]                     # 0.5, 0.75, 1.0
biomass = wue * W                          # proportional to WUE * water
R = biomass / biomass[ref]                 # relative growth (control = 1)

# --- q per level and fits (control point excluded: ln W = 0) --------------
mask = W < 1
x, y = np.log(W[mask]), np.log(R[mask])
print("per-level q:", np.round(y / x, 2))

q_log = np.sum(x * y) / np.sum(x ** 2)                       # log-space, through origin
q_nls, cov = curve_fit(lambda w, q: w ** q, W[mask], R[mask], p0=[1.0])
print(f"q (log-space)  = {q_log:.2f}")
print(f"q (nonlinear)  = {q_nls[0]:.2f} +/- {np.sqrt(cov[0, 0]):.2f}")

# --- decomposition: q = e + w (e from water supplied, w from WUE) ---------
w_exp = np.sum(x * np.log(wue[mask] / wue[ref])) / np.sum(x ** 2)
print(f"w (WUE exponent) = {w_exp:.2f}, e (water exponent) = 1.00 by construction")

# --- Monte Carlo: how much do the WUE SDs move q? -------------------------
rng = np.random.default_rng(1)
qs = []
for _ in range(5000):
    s = rng.normal(wue, wue_sd)
    b = s * W
    r = (b / b[ref])[mask]
    qs.append(np.sum(x * np.log(r)) / np.sum(x ** 2))
print(f"q Monte Carlo: mean {np.mean(qs):.2f}, 95% interval "
      f"[{np.percentile(qs, 2.5):.2f}, {np.percentile(qs, 97.5):.2f}]")

per-level q: [0.95 1.48]
q (log-space)  = 1.03
q (nonlinear)  = 1.09 +/- 0.24
w (WUE exponent) = 0.03, e (water exponent) = 1.00 by construction
q Monte Carlo: mean 1.02, 95% interval [0.77, 1.29]


In [6]:
"""Drought-sensitivity exponent q (R = W**q) for cocksfoot cultivars (Dactylis glomerata).

Inputs: total dry matter yield (g DM m-2) and WUE (g DM L-1) at FC80, FC60, FC40.
W is relative to FC80 (1.0, 0.75, 0.5).
  q = exponent of yield   (R = yield / yield at FC80)
  w = exponent of WUE     (WUE / WUE at FC80)
  e = q - w               (implied exponent of water use, since yield = WUE * water)
"""
import numpy as np
from scipy.optimize import curve_fit

FC = np.array([80.0, 60.0, 40.0])
W = FC / FC[0]                      # 1.0, 0.75, 0.5
cultivars = ["Amba", "Laban", "RGT Lovely", "Prolana", "Tandem", "Mean"]

# yield (g DM m-2): mean, SD at FC80, FC60, FC40
yld = np.array([[649.9, 603.1, 619.3], [588.2, 552.4, 549.2], [692.6, 653.8, 568.8],
                [642.2, 687.5, 632.9], [666.4, 575.3, 573.8], [647.9, 614.4, 588.8]])
yld_sd = np.array([[34.3, 63.9, 60.0], [65.5, 85.7, 29.5], [102.0, 83.0, 57.5],
                   [50.8, 64.6, 81.5], [85.0, 57.6, 30.2], [75.2, 83.7, 75.2]])
# WUE (g DM L-1)
wue = np.array([[1.066, 1.176, 1.356], [1.122, 1.195, 1.375], [1.161, 1.202, 1.330],
                [1.056, 1.169, 1.332], [1.235, 1.294, 1.409], [1.128, 1.207, 1.360]])

x = np.log(W[1:])                   # control point (ln W = 0) adds nothing to the fit

def exponent(values):
    """Log-space regression through the origin of ln(value/value_FC80) on ln W."""
    y = np.log(values[1:] / values[0])
    return np.sum(x * y) / np.sum(x ** 2)

def exponent_nls(values):
    r = values[1:] / values[0]
    q, _ = curve_fit(lambda w, q: w ** q, W[1:], r, p0=[0.3])
    return q[0]

rng = np.random.default_rng(1)
print(f"{'cultivar':<11}{'q_yield':>8}{'q_NLS':>7}{'w_WUE':>7}{'e=q-w':>7}"
      f"{'  q 95% interval':>20}{'  water use L/m2 (FC80,60,40)':>32}")
for i, name in enumerate(cultivars):
    q, qn, w = exponent(yld[i]), exponent_nls(yld[i]), exponent(wue[i])
    # Monte Carlo: SD of the replicates is used as the SE (deliberately conservative)
    qs = [exponent(rng.normal(yld[i], yld_sd[i])) for _ in range(5000)]
    lo, hi = np.percentile(qs, [2.5, 97.5])
    water = yld[i] / wue[i]                       # implied water use, L m-2
    print(f"{name:<11}{q:8.2f}{qn:7.2f}{w:7.2f}{q - w:7.2f}"
          f"{f'  [{lo:.2f}, {hi:.2f}]':>20}{'  ' + ', '.join(f'{v:.0f}' for v in water):>32}")

cultivar    q_yield  q_NLS  w_WUE  e=q-w      q 95% interval     water use L/m2 (FC80,60,40)
Amba           0.10   0.10  -0.35   0.44       [-0.20, 0.43]                   610, 513, 457
Laban          0.12   0.12  -0.28   0.40       [-0.36, 0.52]                   524, 462, 399
RGT Lovely     0.27   0.27  -0.18   0.46       [-0.37, 0.80]                   597, 544, 428
Prolana       -0.02  -0.02  -0.34   0.32       [-0.43, 0.42]                   608, 588, 475
Tandem         0.26   0.27  -0.19   0.45       [-0.26, 0.69]                   540, 445, 407
Mean           0.14   0.15  -0.26   0.41       [-0.39, 0.66]                   574, 509, 433


In [7]:
"""Drought-sensitivity exponent q (R = W**q) for Eragrostis lehmanniana from Snyman et al. (1997).

Relative transpiration (Fig 1b) and relative WUE (Fig 2b) come from the fitted equations in the
figure captions, with x = |leaf water potential| in kPa (the printed Fig 2b equation is garbled;
this is the reconstruction that matches the plotted curve).
Relative photosynthesis ~ relative transpiration * relative WUE  (since WUE = F / transpiration).
W is a 0-1 rescaling of leaf water potential: W = (psi_dry - |psi|) / (psi_dry - psi_wet).
"""
import numpy as np
from scipy.optimize import curve_fit

def rel_transpiration(x):                     # Fig 1b, fraction of reference
    return (52.4281 + 5.4403 * np.log(3000.03 - (x - 3000.01))) / 100

def rel_wue(x):                               # Fig 2b, fraction of reference
    return 10.7557 * np.exp(0.000492 * (3000.03 - (x - 3000.01))) / 100

def fit_q(W, R):
    x, y = np.log(W), np.log(R)
    q_log = np.sum(x * y) / np.sum(x ** 2)                       # ln R = q ln W, through origin
    q_nls, cov = curve_fit(lambda w, q: w ** q, W, R, p0=[1.0])  # R = W**q on original scale
    return q_log, q_nls[0], np.sqrt(cov[0, 0])

def run(psi_wet=1350.0, psi_dry=6000.0, psi_points=(3000, 4000, 5000)):
    psi = np.array(psi_points, float)
    W = (psi_dry - psi) / (psi_dry - psi_wet)
    R = rel_transpiration(psi) * rel_wue(psi)
    return psi, W, R

# --- base case: the three points used earlier ------------------------------
psi, W, R = run()
print("psi(kPa)    W   rel.transp  rel.WUE      R   q(level)")
for p, w, r in zip(psi, W, R):
    print(f"{p:7.0f} {w:6.3f} {rel_transpiration(p):10.3f} {rel_wue(p):8.3f} {r:6.3f} {np.log(r)/np.log(w):8.2f}")
ql, qn, se = fit_q(W, R)
print(f"\nBase case: q (log) = {ql:.2f}, q (NLS) = {qn:.2f} +/- {se:.2f}")

# --- denser grid: every 250 kPa from -2000 to -5500 -------------------------
psi, W, R = run(psi_points=np.arange(2000, 5501, 250))
ql, qn, se = fit_q(W, R)
print(f"Dense grid (n={len(W)}): q (log) = {ql:.2f}, q (NLS) = {qn:.2f} +/- {se:.2f}")

# --- sensitivity to how W is anchored (this is the weakest assumption) ------
print("\nSensitivity to W anchors (dense grid):")
for wet, dry in [(1350, 6000), (1350, 6500), (1000, 6000), (2000, 6000), (2433, 6000)]:
    psi, W, R = run(wet, dry, np.arange(2500, 5501, 250))
    m = (W > 0) & (W < 1)
    print(f"  wet={wet:5d}, dry={dry:5d}:  q (log) = {fit_q(W[m], R[m])[0]:.2f}")

# --- alternative: authors state photosynthesis declines LINEARLY with potential ---
print("\nIf relative photosynthesis is linear in psi (Venter et al. 1997), R = W and q = 1.00")

psi(kPa)    W   rel.transp  rel.WUE      R   q(level)
   3000  0.645      0.960    0.471  0.452     1.81
   4000  0.430      0.938    0.288  0.270     1.55
   5000  0.215      0.900    0.176  0.158     1.20

Base case: q (log) = 1.31, q (NLS) = 1.58 +/- 0.18
Dense grid (n=15): q (log) = 1.21, q (NLS) = 1.61 +/- 0.08

Sensitivity to W anchors (dense grid):
  wet= 1350, dry= 6000:  q (log) = 1.21
  wet= 1350, dry= 6500:  q (log) = 1.54
  wet= 1000, dry= 6000:  q (log) = 1.15
  wet= 2000, dry= 6000:  q (log) = 1.34
  wet= 2433, dry= 6000:  q (log) = 1.44

If relative photosynthesis is linear in psi (Venter et al. 1997), R = W and q = 1.00


In [8]:
"""Drought-sensitivity exponent q (R = W**q) for Lupinus albus (dehydration escape).

Data: dry matter and rainfall in wetter vs drier years.
  Wetter (1987-88, 1989-90): 1810 g/m2 at 687.5 mm
  Drier  (1990-91, 1991-92):  570 g/m2 at 513.5 mm
W = effective rainfall relative to the wetter years, R = dry matter relative to the wetter years.
L is a fixed rainfall loss (runoff + soil evaporation, mm) subtracted before scaling.
"""
import warnings
import numpy as np
from scipy.optimize import curve_fit

warnings.filterwarnings("ignore", message="Covariance of the parameters")  # 1 point, 1 parameter
rain = np.array([687.5, 513.5])    # mm (wetter, drier)
dm = np.array([1810.0, 570.0])     # g DM / m2

def q_for(L=0.0):
    eff = rain - L
    W = eff / eff[0]
    R = dm / dm[0]
    # with a single stressed point both estimators reduce to ln R / ln W
    q_log = np.log(R[1]) / np.log(W[1])
    q_nls, _ = curve_fit(lambda w, q: w ** q, W[1:], R[1:], p0=[1.0], maxfev=10000)
    return W[1], R[1], q_log, q_nls[0]

# --- base case: all rainfall counted ----------------------------------------
W, R, ql, qn = q_for(0)
print(f"Productivity: wet = {dm[0]/rain[0]:.2f}, dry = {dm[1]/rain[1]:.2f} g DM per L rainfall "
      "(1 mm = 1 L m-2)")
print(f"Base case (L = 0): W = {W:.3f}, R = {R:.3f}, q (log) = {ql:.2f}, q (NLS) = {qn:.2f}")

# --- decomposition q = e + w (e = 1 for rainfall supply, w from productivity per litre) ---
w_exp = np.log((dm[1] / rain[1]) / (dm[0] / rain[0])) / np.log(rain[1] / rain[0])
print(f"Decomposition: e = 1.00 (rainfall), w = {w_exp:.2f}, q = e + w = {1 + w_exp:.2f}")

# --- sensitivity to how much rainfall never reaches the roots ----------------
print("\nSensitivity to fixed rainfall loss L:")
print("  L (mm)      W      q")
for L in [0, 100, 200, 250, 300, 350]:
    W, R, ql, _ = q_for(L)
    print(f"  {L:5d}  {W:7.3f}  {ql:5.2f}")

# --- sensitivity to proportional runoff does nothing: q is unchanged ---------
print("\nNote: a constant PROPORTIONAL loss (e.g. 20% runoff) cancels out of W and leaves q unchanged.")

Productivity: wet = 2.63, dry = 1.11 g DM per L rainfall (1 mm = 1 L m-2)
Base case (L = 0): W = 0.747, R = 0.315, q (log) = 3.96, q (NLS) = 3.96
Decomposition: e = 1.00 (rainfall), w = 2.96, q = e + w = 3.96

Sensitivity to fixed rainfall loss L:
  L (mm)      W      q
      0    0.747   3.96
    100    0.704   3.29
    200    0.643   2.62
    250    0.602   2.28
    300    0.551   1.94
    350    0.484   1.59

Note: a constant PROPORTIONAL loss (e.g. 20% runoff) cancels out of W and leaves q unchanged.
